In [1]:
# Install dependencies
# %pip install pandas matplotlib scikit-learn imbalanced-learn keras tensorflow --quiet

from features import get_train_test_data

data = get_train_test_data("data/bank_data_train.csv", sampler="none", num_impute="mean", cat_handle="drop")

print(f"x_train shape: {data.x_train.shape}")
print(f"x_test shape: {data.x_test.shape}")
print(f"y_train shape: {data.y_train.shape}")
print(f"y_test shape: {data.y_test.shape}")

# Display first 5 rows of x_train
data.x_train.head()

x_train shape: (284152, 113)
x_test shape: (71038, 113)
y_train shape: (284152,)
y_test shape: (71038,)


,CR_PROD_CNT_IL,AMOUNT_RUB_CLO_PRC,PRC_ACCEPTS_A_EMAIL_LINK,APP_REGISTR_RGN_CODE,PRC_ACCEPTS_A_POS,PRC_ACCEPTS_A_TK,TURNOVER_DYNAMIC_IL_1M,CNT_TRAN_AUT_TENDENCY1M,SUM_TRAN_AUT_TENDENCY1M,AMOUNT_RUB_SUP_PRC,...,PACK_103,PACK_104,PACK_105,PACK_107,PACK_108,PACK_109,PACK_301,PACK_K01,PACK_M01,PACK_O01
54105,2.061893,-0.018943,0.0,3.479916e-01,0.0,0.0,-0.044935,-0.997829,-1.489598e+00,1.833618,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
313302,-0.244365,-0.430039,0.0,-7.878802e-16,0.0,0.0,-0.044935,0.000000,-3.523313e-16,-0.513119,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
302648,-0.244365,-0.430039,0.0,-7.878802e-16,0.0,0.0,-0.044935,0.000000,-3.523313e-16,-0.634595,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
350887,2.061893,0.246228,0.0,3.479916e-01,0.0,0.0,-0.044935,0.000000,-3.523313e-16,0.393918,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41493,-0.244365,-0.385170,0.0,3.479916e-01,0.0,0.0,-0.044935,3.966221,3.721238e+00,0.638030,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [2]:
# print name of features used for training
print(f"Features used for training: {data.x_train.columns.tolist()}")

Features used for training: ['CR_PROD_CNT_IL', 'AMOUNT_RUB_CLO_PRC', 'PRC_ACCEPTS_A_EMAIL_LINK', 'APP_REGISTR_RGN_CODE', 'PRC_ACCEPTS_A_POS', 'PRC_ACCEPTS_A_TK', 'TURNOVER_DYNAMIC_IL_1M', 'CNT_TRAN_AUT_TENDENCY1M', 'SUM_TRAN_AUT_TENDENCY1M', 'AMOUNT_RUB_SUP_PRC', 'PRC_ACCEPTS_A_AMOBILE', 'SUM_TRAN_AUT_TENDENCY3M', 'PRC_ACCEPTS_TK', 'PRC_ACCEPTS_A_MTP', 'REST_DYNAMIC_FDEP_1M', 'CNT_TRAN_AUT_TENDENCY3M', 'CNT_ACCEPTS_TK', 'REST_DYNAMIC_SAVE_3M', 'CR_PROD_CNT_VCU', 'REST_AVG_CUR', 'CNT_TRAN_MED_TENDENCY1M', 'AMOUNT_RUB_NAS_PRC', 'TRANS_COUNT_SUP_PRC', 'CNT_TRAN_CLO_TENDENCY1M', 'SUM_TRAN_MED_TENDENCY1M', 'PRC_ACCEPTS_A_ATM', 'PRC_ACCEPTS_MTP', 'TRANS_COUNT_NAS_PRC', 'CNT_ACCEPTS_MTP', 'CR_PROD_CNT_TOVR', 'CR_PROD_CNT_PIL', 'SUM_TRAN_CLO_TENDENCY1M', 'TURNOVER_CC', 'TRANS_COUNT_ATM_PRC', 'AMOUNT_RUB_ATM_PRC', 'TURNOVER_PAYM', 'AGE', 'CNT_TRAN_MED_TENDENCY3M', 'CR_PROD_CNT_CC', 'SUM_TRAN_MED_TENDENCY3M', 'REST_DYNAMIC_FDEP_3M', 'REST_DYNAMIC_IL_1M', 'SUM_TRAN_CLO_TENDENCY3M', 'LDEAL_TENOR_M

In [4]:
# keras neural network classifier
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam

model = Sequential(
    [
        Dense(
            512,
            activation="relu",
            input_dim=data.x_train.shape[1],
            kernel_regularizer=keras.regularizers.l2(1e-9),
        ),
        Dropout(0.1),
        Dense(512, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-9)),
        Dropout(0.1),
        Dense(1, activation="sigmoid", kernel_regularizer=keras.regularizers.l2(1e-9)),
    ]
)

import keras
from keras.callbacks import Callback


class StopAtAUC(Callback):
    def __init__(self, target_auc=0.85, monitor="val_auc"):
        super().__init__()
        self.target_auc = target_auc
        self.monitor = monitor

    def on_epoch_end(self, _, logs=None):
        current_auc = logs.get(self.monitor)
        current_auc = float(current_auc) if current_auc is not None else None
        if current_auc is not None and current_auc >= self.target_auc:
            print(
                f"\n\nReached target AUC of {self.target_auc:.4f}! Stopping training."
            )
            self.model.stop_training = True


stop_at_auc = StopAtAUC()

LOSS_FUNCTION = "binary_crossentropy"
LEARNING_RATE = 0.001
EPOCHS = 60
BATCH_SIZE = 1024

model.compile(
    loss=LOSS_FUNCTION,
    optimizer=Adam(learning_rate=LEARNING_RATE),
    metrics=[keras.metrics.AUC(name="val_auc")],
)

model.fit(
    data.x_train,
    data.y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[stop_at_auc],
    verbose=1,
    validation_data=(data.x_test, data.y_test),
)

/home/samy/churn/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/60


I0000 00:00:1779019357.429181    2480 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_20042__.20


269/278 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2907 - val_auc: 0.6521

I0000 00:00:1779019361.175004    2480 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_20042__.20


278/278 ━━━━━━━━━━━━━━━━━━━━ 11s 26ms/step - loss: 0.2644 - val_auc: 0.7124 - val_loss: 0.2494 - val_val_auc: 0.7600
Epoch 2/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.2443 - val_auc: 0.7739 - val_loss: 0.2431 - val_val_auc: 0.7771
Epoch 3/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.2384 - val_auc: 0.7907 - val_loss: 0.2398 - val_val_auc: 0.7867
Epoch 4/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.2340 - val_auc: 0.8028 - val_loss: 0.2390 - val_val_auc: 0.7911
Epoch 5/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.2308 - val_auc: 0.8114 - val_loss: 0.2370 - val_val_auc: 0.7973
Epoch 6/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.2283 - val_auc: 0.8177 - val_loss: 0.2386 - val_val_auc: 0.7939
Epoch 7/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.2265 - val_auc: 0.8223 - val_loss: 0.2347 - val_val_auc: 0.8028
Epoch 8/60
278/278 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.2241 - val_auc: 0.8279 - val_loss: 0.2334 - val_val_auc: 0.8075


In [ ]:
LIBRARYNAME = "keras"
ALGORITHMNAME = "neural_network"

metrics = model.get_metrics_result()
AUC = metrics['auc']
LOSS = metrics['loss']

import pandas as pd

print(model.summary())

pd.DataFrame({
    'Library': [LIBRARYNAME],
    'Algorithm': [ALGORITHMNAME],
    'LossFunction': [LOSS_FUNCTION],
    'LearningRate': [LEARNING_RATE],
    'Epochs': [EPOCHS],
    'BatchSize': [BATCH_SIZE],
    'AUC': [AUC],
    'Loss': [LOSS]
})

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,013 (74.27 KB)

 Trainable params: 6,337 (24.75 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 12,676 (49.52 KB)

None


,Library,Algorithm,LossFunction,LearningRate,Epochs,BatchSize,AUC,Loss
0,keras,neural_network,binary_crossentropy,0.001,300,1024,0.849541,0.213481


In [ ]:
from features import get_test_data

selected_features = data.x_train.columns.tolist()
df = get_test_data("data/bank_data_test.csv", data)
test_df = df[selected_features]

predictions = model.predict(test_df)
predictions

2775/2775 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step


array([[0.17869318],
       [0.00529136],
       [0.01792705],
       ...,
       [0.0342129 ],
       [0.00266396],
       [0.41390225]], shape=(88798, 1), dtype=float32)

In [ ]:
# save probabilities to csv file with ID and TARGET column
import pandas as pd

output_df = pd.DataFrame({
    'ID': df['ID'],
    'TARGET': predictions.flatten()
})
output_df.to_csv("churn_keras_predictions.csv", index=False)
output_df.head()

,ID,TARGET
0,400980,0.178693
1,525062,0.005291
2,280316,0.017927
3,496066,0.024806
4,375031,0.000932
